In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Hospital Analytics").getOrCreate()

---

# Hospital Patient, Doctor, Appointment, Billing Analytics

## Step 1 — Create Data in Google Colab

```python
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""
```

```python
doctors_csv = """doctor_id,doctor_name,specialization,city,consultation_fee
101,Dr Sharma,Cardiology,Hyderabad,1200
102,Dr Iyer,Dermatology,Bangalore,800
103,Dr Khan,Orthopedics,Mumbai,1500
104,Dr Reddy,Pediatrics,Delhi,900
105,Dr Mehta,Neurology,Hyderabad,2000
106,Dr Nair,Cardiology,Chennai,1300
107,Dr Verma,Dermatology,Pune,850
108,Dr Rao,Orthopedics,Delhi,1400
"""
```

```python
appointments_csv = """appointment_id,patient_id,doctor_id,appointment_date,status
1,1,101,2024-03-01,Completed
2,2,102,2024-03-01,Completed
3,3,103,2024-03-02,Completed
4,4,104,2024-03-02,Pending
5,5,106,2024-03-03,Completed
6,6,105,2024-03-03,Completed
7,7,107,2024-03-04,Cancelled
8,8,108,2024-03-04,Completed
9,9,102,2024-03-05,Completed
10,10,103,2024-03-05,Completed
11,11,101,2024-03-06,Pending
12,12,104,2024-03-06,Completed
13,1,105,2024-03-07,Completed
14,3,108,2024-03-07,Completed
15,6,101,2024-03-08,Cancelled
16,9,106,2024-03-08,Completed
"""
```

```python
bills_csv = """bill_id,appointment_id,bill_amount,payment_mode,payment_status
1,1,1200,UPI,Paid
2,2,800,Credit Card,Paid
3,3,1500,Cash,Paid
4,4,900,UPI,Pending
5,5,1300,Debit Card,Paid
6,6,2000,Credit Card,Paid
7,7,850,Cash,Cancelled
8,8,1400,UPI,Paid
9,9,800,UPI,Paid
10,10,1500,Credit Card,Paid
11,11,1200,UPI,Pending
12,12,900,Cash,Paid
13,13,2000,Credit Card,Paid
14,14,1400,UPI,Paid
15,15,1200,Cash,Cancelled
16,16,1300,Debit Card,Paid
"""
```

```python
hospital_logs_txt = """Aarav login
Priya login
Rahul appointment
Sneha login
Aarav payment
Kiran appointment
Meera login
Vikram appointment
Divya payment
Farhan login
Simran appointment
Neha payment
Amit login
Rahul payment
Meera appointment
Aarav logout
Priya payment
Divya login
Vikram payment
Farhan appointment
"""
```

```python
patient_profiles_json = """[
 {
  "patient_id": 1,
  "name": "Aarav",
  "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
  "allergies": ["Dust", "Peanuts"]
 },
 {
  "patient_id": 2,
  "name": "Priya",
  "contact": {"email": "priya@mail.com", "phone": "9000022222"},
  "allergies": ["Pollen"]
 },
 {
  "patient_id": 3,
  "name": "Rahul",
  "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
  "allergies": ["Dust", "Milk"]
 },
 {
  "patient_id": 6,
  "name": "Meera",
  "contact": {"email": "meera@mail.com", "phone": "9000066666"},
  "allergies": ["Seafood"]
 },
 {
  "patient_id": 10,
  "name": "Vikram",
  "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
  "allergies": ["Pollen", "Dust"]
 }
]
"""
```
```
with open("patients.csv", "w") as f:
 f.write(patients_csv)

with open("patients.csv", "w") as f:
    f.write(patients_csv)

with open("doctors.csv", "w") as f:
    f.write(doctors_csv)

with open("appointments.csv", "w") as f:
    f.write(appointments_csv)

with open("bills.csv", "w") as f:
    f.write(bills_csv)

with open("hospital_logs.txt", "w") as f:
    f.write(hospital_logs_txt)

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)

print("Hospital datasets created successfully")
```


In [1]:
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""

doctors_csv = """doctor_id,doctor_name,specialization,city,consultation_fee
101,Dr Sharma,Cardiology,Hyderabad,1200
102,Dr Iyer,Dermatology,Bangalore,800
103,Dr Khan,Orthopedics,Mumbai,1500
104,Dr Reddy,Pediatrics,Delhi,900
105,Dr Mehta,Neurology,Hyderabad,2000
106,Dr Nair,Cardiology,Chennai,1300
107,Dr Verma,Dermatology,Pune,850
108,Dr Rao,Orthopedics,Delhi,1400
"""

appointments_csv = """appointment_id,patient_id,doctor_id,appointment_date,status
1,1,101,2024-03-01,Completed
2,2,102,2024-03-01,Completed
3,3,103,2024-03-02,Completed
4,4,104,2024-03-02,Pending
5,5,106,2024-03-03,Completed
6,6,105,2024-03-03,Completed
7,7,107,2024-03-04,Cancelled
8,8,108,2024-03-04,Completed
9,9,102,2024-03-05,Completed
10,10,103,2024-03-05,Completed
11,11,101,2024-03-06,Pending
12,12,104,2024-03-06,Completed
13,1,105,2024-03-07,Completed
14,3,108,2024-03-07,Completed
15,6,101,2024-03-08,Cancelled
16,9,106,2024-03-08,Completed
"""

bills_csv = """bill_id,appointment_id,bill_amount,payment_mode,payment_status
1,1,1200,UPI,Paid
2,2,800,Credit Card,Paid
3,3,1500,Cash,Paid
4,4,900,UPI,Pending
5,5,1300,Debit Card,Paid
6,6,2000,Credit Card,Paid
7,7,850,Cash,Cancelled
8,8,1400,UPI,Paid
9,9,800,UPI,Paid
10,10,1500,Credit Card,Paid
11,11,1200,UPI,Pending
12,12,900,Cash,Paid
13,13,2000,Credit Card,Paid
14,14,1400,UPI,Paid
15,15,1200,Cash,Cancelled
16,16,1300,Debit Card,Paid
"""

hospital_logs_txt = """Aarav login
Priya login
Rahul appointment
Sneha login
Aarav payment
Kiran appointment
Meera login
Vikram appointment
Divya payment
Farhan login
Simran appointment
Neha payment
Amit login
Rahul payment
Meera appointment
Aarav logout
Priya payment
Divya login
Vikram payment
Farhan appointment
"""

patient_profiles_json = """[
  {
    "patient_id": 1,
    "name": "Aarav",
    "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
    "allergies": ["Dust", "Peanuts"]
  },
  {
    "patient_id": 2,
    "name": "Priya",
    "contact": {"email": "priya@mail.com", "phone": "9000022222"},
    "allergies": ["Pollen"]
  },
  {
    "patient_id": 3,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
    "allergies": ["Dust", "Milk"]
  },
  {
    "patient_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "allergies": ["Seafood"]
  },
  {
    "patient_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "allergies": ["Pollen", "Dust"]
  }
]
"""

with open("patients.csv", "w") as f:
    f.write(patients_csv)

with open("doctors.csv", "w") as f:
    f.write(doctors_csv)

with open("appointments.csv", "w") as f:
    f.write(appointments_csv)

with open("bills.csv", "w") as f:
    f.write(bills_csv)

with open("hospital_logs.txt", "w") as f:
    f.write(hospital_logs_txt)

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)

print("Hospital datasets created successfully")

Hospital datasets created successfully


---

## Step 2 — Load Data

```python
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patient_profiles.json", multiLine=True)
```

---

In [3]:
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patient_profiles.json", multiLine=True)

---

# 100 PySpark Exercises

## Section 1 — DataFrame Basics

1. Show all patients
2. Show all doctors
3. Show all appointments
4. Show all bills
5. Print schema of patients
6. Print schema of doctors
7. Print schema of appointments
8. Count total patients
9. Count total doctors
10. Show first 5 appointments

---


In [4]:
patients.show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [5]:
doctors.show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+



In [6]:
appointments.show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [7]:
bills.show()

+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|      1|             1|       1200|         UPI|          Paid|
|      2|             2|        800| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|      4|             4|        900|         UPI|       Pending|
|      5|             5|       1300|  Debit Card|          Paid|
|      6|             6|       2000| Credit Card|          Paid|
|      7|             7|        850|        Cash|     Cancelled|
|      8|             8|       1400|         UPI|          Paid|
|      9|             9|        800|         UPI|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|     11|            11|       1200|         UPI|       Pending|
|     12|            12|        900|        Cash|          Paid|
|     13|            13| 

In [8]:
patients.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registration_date: date (nullable = true)



In [9]:
doctors.printSchema()

root
 |-- doctor_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- consultation_fee: integer (nullable = true)



In [10]:
appointments.printSchema()

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- status: string (nullable = true)



In [12]:
patients.count()

12

In [13]:
doctors.count()

8

In [14]:
appointments.show(5)

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
+--------------+----------+---------+----------------+---------+
only showing top 5 rows


---

## Section 2 — Select, Rename, Filter

11. Select patient name, city, age
12. Select doctor name, specialization, consultation fee
13. Rename name → patient_name
14. Rename doctor_name → consultant_name
15. Show patients from Hyderabad
16. Show female patients
17. Show patients older than 35
18. Show doctors from Hyderabad
19. Show Cardiology doctors
20. Show doctors with fee > 1000

---

In [15]:
patients.select("name","city","age").show()

+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+



In [17]:
doctors.select("doctor_name","specialization","consultation_fee").show()

+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+



In [18]:
patients.withColumnRenamed("name","patient_name").show()

+----------+------------+---------+---+------+-----------------+
|patient_id|patient_name|     city|age|gender|registration_date|
+----------+------------+---------+---+------+-----------------+
|         1|       Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2|       Priya|Bangalore| 34|Female|       2023-02-12|
|         3|       Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4|       Sneha|    Delhi| 26|Female|       2023-04-15|
|         5|       Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6|       Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|        Amit|     Pune| 45|  Male|       2023-06-22|
|         8|        Neha|    Delhi| 28|Female|       2023-07-10|
|         9|       Divya|Bangalore| 33|Female|       2023-07-15|
|        10|      Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|      Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|      Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------------+

In [19]:
doctors.withColumnRenamed("doctor_name","consultant_name").show()

+---------+---------------+--------------+---------+----------------+
|doctor_id|consultant_name|specialization|     city|consultation_fee|
+---------+---------------+--------------+---------+----------------+
|      101|      Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|        Dr Iyer|   Dermatology|Bangalore|             800|
|      103|        Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104|       Dr Reddy|    Pediatrics|    Delhi|             900|
|      105|       Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|        Dr Nair|    Cardiology|  Chennai|            1300|
|      107|       Dr Verma|   Dermatology|     Pune|             850|
|      108|         Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+---------------+--------------+---------+----------------+



In [21]:
patients.filter(patients.city=="Hyderabad").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [23]:
patients.filter(patients.gender=="Female").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [24]:
patients.filter(patients.age>35).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [25]:
doctors.filter(doctors.city=="Hyderabad").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
+---------+-----------+--------------+---------+----------------+



In [26]:
doctors.filter(doctors.specialization=="Cardiology").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
+---------+-----------+--------------+---------+----------------+



In [28]:
doctors.filter(doctors.consultation_fee>1000).show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+



---

## Section 3 — Sorting and Limit

21. Sort patients by age (asc)
22. Sort patients by age (desc)
23. Top 5 oldest patients
24. Youngest 3 patients
25. Sort doctors by fee desc
26. Top 3 highest fee doctors
27. Lowest fee doctors
28. Sort appointments by date
29. Sort bills by amount desc
30. Top 5 highest bills

---


In [29]:
patients.sort(patients.age).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+



In [31]:
patients.sort(patients.age.desc()).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [32]:
patients.sort(patients.age.desc()).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
+----------+------+---------+---+------+-----------------+
only showing top 5 rows


In [33]:
patients.sort(patients.age).show(3)

+----------+------+-----+---+------+-----------------+
|patient_id|  name| city|age|gender|registration_date|
+----------+------+-----+---+------+-----------------+
|        12|Simran|Delhi| 25|Female|       2023-08-21|
|         4| Sneha|Delhi| 26|Female|       2023-04-15|
|         8|  Neha|Delhi| 28|Female|       2023-07-10|
+----------+------+-----+---+------+-----------------+
only showing top 3 rows


In [34]:
doctors.sort(doctors.consultation_fee.desc()).show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
+---------+-----------+--------------+---------+----------------+



In [35]:
doctors.sort(doctors.consultation_fee.desc()).show(5)

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
+---------+-----------+--------------+---------+----------------+
only showing top 5 rows


In [36]:
doctors.sort(doctors.consultation_fee).show(3)

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
+---------+-----------+--------------+---------+----------------+
only showing top 3 rows


In [38]:
appointments.sort(appointments.appointment_date).show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [40]:
bills.sort(bills.bill_amount.desc()).show()

+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|      6|             6|       2000| Credit Card|          Paid|
|     13|            13|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
|     14|            14|       1400|         UPI|          Paid|
|      5|             5|       1300|  Debit Card|          Paid|
|     16|            16|       1300|  Debit Card|          Paid|
|      1|             1|       1200|         UPI|          Paid|
|     11|            11|       1200|         UPI|       Pending|
|     15|            15|       1200|        Cash|     Cancelled|
|      4|             4|        900|         UPI|       Pending|
|     12|            12| 

In [41]:
bills.sort(bills.bill_amount.desc()).show(5)

+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|     13|            13|       2000| Credit Card|          Paid|
|      6|             6|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
+-------+--------------+-----------+------------+--------------+
only showing top 5 rows


---

## Section 4 — Aggregations

31. Count patients by city
32. Count patients by gender
33. Count doctors by specialization
34. Average patient age
35. Max patient age
36. Min patient age
37. Avg consultation fee
38. Max consultation fee
39. Total bill amount
40. Avg bill amount

---

In [42]:
patients.groupBy(patients.city).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [43]:
patients.groupBy(patients.gender).count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+



In [46]:
doctors.groupBy(doctors.specialization).count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+



In [51]:
from pyspark.sql.functions import avg,max,min,sum

In [49]:
patients.select(
    avg(patients.age).alias("AveragePatientAge"),
    max(patients.age).alias("MaxPatientAge"),
    min(patients.age).alias("MinPatientAge")
).show()

+-----------------+-------------+-------------+
|AveragePatientAge|MaxPatientAge|MinPatientAge|
+-----------------+-------------+-------------+
|             35.0|           52|           25|
+-----------------+-------------+-------------+



In [50]:
doctors.select(
    avg(doctors.consultation_fee).alias("AvgConsultationFee"),
    max(doctors.consultation_fee).alias("MaxConsultationFee")
).show()

+------------------+------------------+
|AvgConsultationFee|MaxConsultationFee|
+------------------+------------------+
|           1243.75|              2000|
+------------------+------------------+



In [52]:
bills.select(
    sum(bills.bill_amount).alias("TotalBillAmount"),
    avg(bills.bill_amount).alias("AvgBillAmount")
).show()

+---------------+-------------+
|TotalBillAmount|AvgBillAmount|
+---------------+-------------+
|          20250|     1265.625|
+---------------+-------------+



---

## Section 5 — GroupBy Analytics

41. Find patient count by city.
42. Find doctor count by city.
43. Find appointment count by status.
44. Find bill amount by payment status.
45. Find total revenue by payment mode.
46. Find average bill amount by payment mode.
47. Find appointment count by doctor_id.
48. Find appointment count by patient_id.
49. Find total bill amount by appointment_id.
50. Find average consultation fee by specialization.

---

In [53]:
patients.groupBy(patients.city).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [55]:
doctors.groupBy(doctors.city).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [56]:
appointments.groupBy(appointments.status).count().show()

+---------+-----+
|   status|count|
+---------+-----+
|Completed|   12|
|Cancelled|    2|
|  Pending|    2|
+---------+-----+



In [57]:
bills.groupBy(bills.payment_status).sum("bill_amount").show()

+--------------+----------------+
|payment_status|sum(bill_amount)|
+--------------+----------------+
|     Cancelled|            2050|
|       Pending|            2100|
|          Paid|           16100|
+--------------+----------------+



In [58]:
bills.groupBy(bills.payment_status).agg(
    sum("bill_amount").alias("TotalBillAmount"),
).show()


+--------------+---------------+
|payment_status|TotalBillAmount|
+--------------+---------------+
|     Cancelled|           2050|
|       Pending|           2100|
|          Paid|          16100|
+--------------+---------------+



In [60]:
bills.groupBy(bills.payment_mode).agg(
    sum("bill_amount").alias("TotalBillAmount"),
    avg("bill_amount").alias("AvgBillAmount")
).show()


+------------+---------------+-------------+
|payment_mode|TotalBillAmount|AvgBillAmount|
+------------+---------------+-------------+
| Credit Card|           6300|       1575.0|
|        Cash|           4450|       1112.5|
|  Debit Card|           2600|       1300.0|
|         UPI|           6900|       1150.0|
+------------+---------------+-------------+



In [61]:
appointments.groupBy(appointments.doctor_id).count().show()

+---------+-----+
|doctor_id|count|
+---------+-----+
|      108|    2|
|      101|    3|
|      103|    2|
|      107|    1|
|      102|    2|
|      105|    2|
|      106|    2|
|      104|    2|
+---------+-----+



In [62]:
appointments.groupBy(appointments.patient_id).count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|        12|    1|
|         1|    2|
|         6|    2|
|         3|    2|
|         5|    1|
|         9|    2|
|         4|    1|
|         8|    1|
|         7|    1|
|        10|    1|
|        11|    1|
|         2|    1|
+----------+-----+



In [68]:
bills.groupBy(bills.appointment_id).agg(
    sum(bills.bill_amount).alias("TotalBillAmount")
).sort(bills.appointment_id).show()

+--------------+---------------+
|appointment_id|TotalBillAmount|
+--------------+---------------+
|             1|           1200|
|             2|            800|
|             3|           1500|
|             4|            900|
|             5|           1300|
|             6|           2000|
|             7|            850|
|             8|           1400|
|             9|            800|
|            10|           1500|
|            11|           1200|
|            12|            900|
|            13|           2000|
|            14|           1400|
|            15|           1200|
|            16|           1300|
+--------------+---------------+



In [69]:
doctors.groupBy(doctors.specialization).agg(
    avg(doctors.consultation_fee).alias("AvgConsultationFee")
).show()

+--------------+------------------+
|specialization|AvgConsultationFee|
+--------------+------------------+
|     Neurology|            2000.0|
|   Dermatology|             825.0|
|    Cardiology|            1250.0|
|    Pediatrics|             900.0|
|   Orthopedics|            1450.0|
+--------------+------------------+



---

## Section 6 — Joins

51. Join patients with appointments using patient_id .
52. Show patient name, city, appointment date, and status.
53. Join doctors with appointments using doctor_id .
54. Show doctor name, specialization, appointment date, and status.
55. Join appointments with bills using appointment_id .
56. Show appointment_id, status, bill amount, and payment status.
57. Join patients, appointments, and doctors.
58. Show patient name, doctor name, specialization, and appointment date.
59. Join patients, appointments, doctors, and bills.
60. Show patient name, doctor name, status, bill amount, and payment mode.

---

In [71]:
patients.join(appointments, patients.patient_id==appointments.patient_id).show()

+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|         1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|         2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|         3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|         4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|         5|      106|      2024-03-03|Completed|
|         6| Meera|Hyder

In [72]:
patients.join(appointments, patients.patient_id==appointments.patient_id).select(patients.name,patients.city,appointments.appointment_date,appointments.status).show()

+------+---------+----------------+---------+
|  name|     city|appointment_date|   status|
+------+---------+----------------+---------+
| Aarav|Hyderabad|      2024-03-01|Completed|
| Priya|Bangalore|      2024-03-01|Completed|
| Rahul|   Mumbai|      2024-03-02|Completed|
| Sneha|    Delhi|      2024-03-02|  Pending|
| Kiran|  Chennai|      2024-03-03|Completed|
| Meera|Hyderabad|      2024-03-03|Completed|
|  Amit|     Pune|      2024-03-04|Cancelled|
|  Neha|    Delhi|      2024-03-04|Completed|
| Divya|Bangalore|      2024-03-05|Completed|
|Vikram|   Mumbai|      2024-03-05|Completed|
|Farhan|Hyderabad|      2024-03-06|  Pending|
|Simran|    Delhi|      2024-03-06|Completed|
| Aarav|Hyderabad|      2024-03-07|Completed|
| Rahul|   Mumbai|      2024-03-07|Completed|
| Meera|Hyderabad|      2024-03-08|Cancelled|
| Divya|Bangalore|      2024-03-08|Completed|
+------+---------+----------------+---------+



In [73]:
doctors.join(appointments, doctors.doctor_id==appointments.doctor_id).show()

+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|appointment_id|patient_id|doctor_id|appointment_date|   status|
+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|             1|         1|      101|      2024-03-01|Completed|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|             2|         2|      102|      2024-03-01|Completed|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|             3|         3|      103|      2024-03-02|Completed|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|             4|         4|      104|      2024-03-02|  Pending|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|             5|         5|      

In [74]:
doctors.join(appointments, doctors.doctor_id==appointments.doctor_id).select(doctors.doctor_name,doctors.specialization,appointments.appointment_date,appointments.status).show()

+-----------+--------------+----------------+---------+
|doctor_name|specialization|appointment_date|   status|
+-----------+--------------+----------------+---------+
|  Dr Sharma|    Cardiology|      2024-03-01|Completed|
|    Dr Iyer|   Dermatology|      2024-03-01|Completed|
|    Dr Khan|   Orthopedics|      2024-03-02|Completed|
|   Dr Reddy|    Pediatrics|      2024-03-02|  Pending|
|    Dr Nair|    Cardiology|      2024-03-03|Completed|
|   Dr Mehta|     Neurology|      2024-03-03|Completed|
|   Dr Verma|   Dermatology|      2024-03-04|Cancelled|
|     Dr Rao|   Orthopedics|      2024-03-04|Completed|
|    Dr Iyer|   Dermatology|      2024-03-05|Completed|
|    Dr Khan|   Orthopedics|      2024-03-05|Completed|
|  Dr Sharma|    Cardiology|      2024-03-06|  Pending|
|   Dr Reddy|    Pediatrics|      2024-03-06|Completed|
|   Dr Mehta|     Neurology|      2024-03-07|Completed|
|     Dr Rao|   Orthopedics|      2024-03-07|Completed|
|  Dr Sharma|    Cardiology|      2024-03-08|Can

In [75]:
appointments.join(bills, appointments.appointment_id==bills.appointment_id).show()

+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|             1|         1|      101|      2024-03-01|Completed|      1|             1|       1200|         UPI|          Paid|
|             2|         2|      102|      2024-03-01|Completed|      2|             2|        800| Credit Card|          Paid|
|             3|         3|      103|      2024-03-02|Completed|      3|             3|       1500|        Cash|          Paid|
|             4|         4|      104|      2024-03-02|  Pending|      4|             4|        900|         UPI|       Pending|
|             5|         5|      106|      2024-03-03|Completed|      5|             5|       1300|  Deb

In [76]:
appointments.join(bills, appointments.appointment_id==bills.appointment_id).select(appointments.appointment_id,appointments.status,bills.bill_amount,bills.payment_status).show()

+--------------+---------+-----------+--------------+
|appointment_id|   status|bill_amount|payment_status|
+--------------+---------+-----------+--------------+
|             1|Completed|       1200|          Paid|
|             2|Completed|        800|          Paid|
|             3|Completed|       1500|          Paid|
|             4|  Pending|        900|       Pending|
|             5|Completed|       1300|          Paid|
|             6|Completed|       2000|          Paid|
|             7|Cancelled|        850|     Cancelled|
|             8|Completed|       1400|          Paid|
|             9|Completed|        800|          Paid|
|            10|Completed|       1500|          Paid|
|            11|  Pending|       1200|       Pending|
|            12|Completed|        900|          Paid|
|            13|Completed|       2000|          Paid|
|            14|Completed|       1400|          Paid|
|            15|Cancelled|       1200|     Cancelled|
|            16|Completed|  

In [77]:
patients.join(appointments, patients.patient_id==appointments.patient_id).join(doctors, doctors.doctor_id==appointments.doctor_id).show()

+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+---------+-----------+--------------+---------+----------------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|doctor_id|doctor_name|specialization|     city|consultation_fee|
+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+---------+-----------+--------------+---------+----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|         1|      101|      2024-03-01|Completed|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|         2|      102|      2024-03-01|Completed|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|            

In [78]:
patients.join(appointments, patients.patient_id==appointments.patient_id).join(doctors, doctors.doctor_id==appointments.doctor_id).select(patients.name,doctors.doctor_name,doctors.specialization,appointments.appointment_date).show()

+------+-----------+--------------+----------------+
|  name|doctor_name|specialization|appointment_date|
+------+-----------+--------------+----------------+
| Aarav|  Dr Sharma|    Cardiology|      2024-03-01|
| Priya|    Dr Iyer|   Dermatology|      2024-03-01|
| Rahul|    Dr Khan|   Orthopedics|      2024-03-02|
| Sneha|   Dr Reddy|    Pediatrics|      2024-03-02|
| Kiran|    Dr Nair|    Cardiology|      2024-03-03|
| Meera|   Dr Mehta|     Neurology|      2024-03-03|
|  Amit|   Dr Verma|   Dermatology|      2024-03-04|
|  Neha|     Dr Rao|   Orthopedics|      2024-03-04|
| Divya|    Dr Iyer|   Dermatology|      2024-03-05|
|Vikram|    Dr Khan|   Orthopedics|      2024-03-05|
|Farhan|  Dr Sharma|    Cardiology|      2024-03-06|
|Simran|   Dr Reddy|    Pediatrics|      2024-03-06|
| Aarav|   Dr Mehta|     Neurology|      2024-03-07|
| Rahul|     Dr Rao|   Orthopedics|      2024-03-07|
| Meera|  Dr Sharma|    Cardiology|      2024-03-08|
| Divya|    Dr Nair|    Cardiology|      2024-

In [80]:
patients.join(appointments, patients.patient_id==appointments.patient_id).join(doctors, doctors.doctor_id==appointments.doctor_id).join(bills,bills.appointment_id==appointments.appointment_id).show()

+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+---------+-----------+--------------+---------+----------------+-------+--------------+-----------+------------+--------------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|doctor_id|doctor_name|specialization|     city|consultation_fee|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+---------+-----------+--------------+---------+----------------+-------+--------------+-----------+------------+--------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|         1|      101|      2024-03-01|Completed|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1|             1|       1200|         UPI|          Paid|
|   

In [81]:
patients.join(appointments, patients.patient_id==appointments.patient_id).join(doctors, doctors.doctor_id==appointments.doctor_id).join(bills,bills.appointment_id==appointments.appointment_id).select(patients.name,doctors.doctor_name,appointments.status,bills.bill_amount,bills.payment_mode).show()

+------+-----------+---------+-----------+------------+
|  name|doctor_name|   status|bill_amount|payment_mode|
+------+-----------+---------+-----------+------------+
| Aarav|  Dr Sharma|Completed|       1200|         UPI|
| Priya|    Dr Iyer|Completed|        800| Credit Card|
| Rahul|    Dr Khan|Completed|       1500|        Cash|
| Sneha|   Dr Reddy|  Pending|        900|         UPI|
| Kiran|    Dr Nair|Completed|       1300|  Debit Card|
| Meera|   Dr Mehta|Completed|       2000| Credit Card|
|  Amit|   Dr Verma|Cancelled|        850|        Cash|
|  Neha|     Dr Rao|Completed|       1400|         UPI|
| Divya|    Dr Iyer|Completed|        800|         UPI|
|Vikram|    Dr Khan|Completed|       1500| Credit Card|
|Farhan|  Dr Sharma|  Pending|       1200|         UPI|
|Simran|   Dr Reddy|Completed|        900|        Cash|
| Aarav|   Dr Mehta|Completed|       2000| Credit Card|
| Rahul|     Dr Rao|Completed|       1400|         UPI|
| Meera|  Dr Sharma|Cancelled|       1200|      

---

## Section 7 — Updating Data with withColumn

61. Add age_group column using age.
62. Add hospital_name = "BotCampus Hospital" .
63. Add fee_with_tax = consultation_fee * 1.18 .
64. Add bill_with_tax = bill_amount * 1.18 .
65. Add bill_in_thousands = bill_amount / 1000 .
66. Add country = "India" to patients.
67. Add doctor_label = doctor_name + " - " + specialization .
68. Add patient_label = name + " - " + city .
69. Add high_bill_flag where bill amount is above 1500.
70. Add senior_patient_flag where age is above 40.

---

In [84]:
from pyspark.sql.functions import *

In [83]:
patients.withColumn("age_group", when(patients.age<30,"Young").when(patients.age<40,"Middle-Aged").otherwise("Old")).show()

+----------+------+---------+---+------+-----------------+-----------+
|patient_id|  name|     city|age|gender|registration_date|  age_group|
+----------+------+---------+---+------+-----------------+-----------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|      Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|Middle-Aged|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|        Old|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|      Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|Middle-Aged|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|Middle-Aged|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|        Old|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|      Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|Middle-Aged|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|        Old|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|Middle-Aged|
|     

In [85]:
patients.withColumn("hospital_name",lit("BotCampus Hospital")).show()

+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     hospital_name|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|BotCampus Hospital|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|BotCampus Hospital|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|BotCampus Hospital|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|BotCampus Hospital|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|BotCampus Hospital|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|BotCampus Hospital|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|BotCampus Hospital|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|BotCampus Hospital|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|BotCampus Hospital|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|BotCam

In [89]:
doctors.withColumn("fee_with_tax", doctors.consultation_fee * lit(1.18)).show()

+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_with_tax|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1416.0|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       944.0|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|      1770.0|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|      1062.0|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|      2360.0|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|      1534.0|
|      107|   Dr Verma|   Dermatology|     Pune|             850|      1003.0|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|      1652.0|
+---------+-----------+--------------+---------+----------------+------------+



In [90]:
bills.withColumn("bill_with_tax", bills.bill_amount * lit(1.18)).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_with_tax|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       1416.0|
|      2|             2|        800| Credit Card|          Paid|        944.0|
|      3|             3|       1500|        Cash|          Paid|       1770.0|
|      4|             4|        900|         UPI|       Pending|       1062.0|
|      5|             5|       1300|  Debit Card|          Paid|       1534.0|
|      6|             6|       2000| Credit Card|          Paid|       2360.0|
|      7|             7|        850|        Cash|     Cancelled|       1003.0|
|      8|             8|       1400|         UPI|          Paid|       1652.0|
|      9|             9|        800|         UPI|          Paid|        944.0|
|     10|            10|       1500| Credit Card|   

In [91]:
bills.withColumn("bill_in_thousands", bills.bill_amount / lit(1000)).show()

+-------+--------------+-----------+------------+--------------+-----------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_in_thousands|
+-------+--------------+-----------+------------+--------------+-----------------+
|      1|             1|       1200|         UPI|          Paid|              1.2|
|      2|             2|        800| Credit Card|          Paid|              0.8|
|      3|             3|       1500|        Cash|          Paid|              1.5|
|      4|             4|        900|         UPI|       Pending|              0.9|
|      5|             5|       1300|  Debit Card|          Paid|              1.3|
|      6|             6|       2000| Credit Card|          Paid|              2.0|
|      7|             7|        850|        Cash|     Cancelled|             0.85|
|      8|             8|       1400|         UPI|          Paid|              1.4|
|      9|             9|        800|         UPI|          Paid|              0.8|
|   

In [93]:
patients.withColumn("country", lit("India")).show()

+----------+------+---------+---+------+-----------------+-------+
|patient_id|  name|     city|age|gender|registration_date|country|
+----------+------+---------+---+------+-----------------+-------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|  India|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  India|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|  India|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|  India|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  India|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|  India|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  India|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  India|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|  India|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  India|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  India|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|  In

In [95]:
doctors.withColumn("doctor_label", concat(doctors.doctor_name,lit(" - "),doctors.specialization)).show()

+---------+-----------+--------------+---------+----------------+--------------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|        doctor_label|
+---------+-----------+--------------+---------+----------------+--------------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|Dr Sharma - Cardi...|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|Dr Iyer - Dermato...|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|Dr Khan - Orthope...|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|Dr Reddy - Pediat...|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|Dr Mehta - Neurology|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|Dr Nair - Cardiology|
|      107|   Dr Verma|   Dermatology|     Pune|             850|Dr Verma - Dermat...|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|Dr Rao - Orthopedics|
+---------+-----------+--------------+-----

In [97]:
patients.withColumn("patient_label", concat(patients.name,lit(" - "),patients.city)).show()

+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     patient_label|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10| Aarav - Hyderabad|
|         2| Priya|Bangalore| 34|Female|       2023-02-12| Priya - Bangalore|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Rahul - Mumbai|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|     Sneha - Delhi|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   Kiran - Chennai|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10| Meera - Hyderabad|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|       Amit - Pune|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|      Neha - Delhi|
|         9| Divya|Bangalore| 33|Female|       2023-07-15| Divya - Bangalore|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Vik

In [98]:
bills.withColumn("high_bill_flag", when(bills.bill_amount>1500,"Yes").otherwise("No")).show()

+-------+--------------+-----------+------------+--------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+-------+--------------+-----------+------------+--------------+--------------+
|      1|             1|       1200|         UPI|          Paid|            No|
|      2|             2|        800| Credit Card|          Paid|            No|
|      3|             3|       1500|        Cash|          Paid|            No|
|      4|             4|        900|         UPI|       Pending|            No|
|      5|             5|       1300|  Debit Card|          Paid|            No|
|      6|             6|       2000| Credit Card|          Paid|           Yes|
|      7|             7|        850|        Cash|     Cancelled|            No|
|      8|             8|       1400|         UPI|          Paid|            No|
|      9|             9|        800|         UPI|          Paid|            No|
|     10|            10|       1500| Cre

In [99]:
patients.withColumn("senior_patient_flag", when(patients.age>40,"Yes").otherwise("No")).show()

+----------+------+---------+---+------+-----------------+-------------------+
|patient_id|  name|     city|age|gender|registration_date|senior_patient_flag|
+----------+------+---------+---+------+-----------------+-------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|                 No|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|                 No|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|                Yes|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|                 No|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|                 No|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|                 No|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|                Yes|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|                 No|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|                 No|
|        10|Vikram|   Mumbai| 52|  Male|       2023-

---

## Section 8 — When / Otherwise

71. Create patient age category: Young, Adult, Senior.
72. Create bill category: High, Medium, Low.
73. Create doctor fee category: Premium, Standard, Basic.
74. Create appointment priority: Pending = High, Completed = Normal, Cancelled = Low.
75. Create city zone: Hyderabad/Bangalore/Chennai = South, Mumbai/Delhi/Pune = Metro.

---

In [100]:
patients.withColumn("age_group", when(patients.age<30,"Young").when(patients.age<40,"Middle-Aged").otherwise("Old")).show()

+----------+------+---------+---+------+-----------------+-----------+
|patient_id|  name|     city|age|gender|registration_date|  age_group|
+----------+------+---------+---+------+-----------------+-----------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|      Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|Middle-Aged|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|        Old|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|      Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|Middle-Aged|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|Middle-Aged|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|        Old|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|      Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|Middle-Aged|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|        Old|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|Middle-Aged|
|     

In [101]:
bills.withColumn("bill_category",when(bills.bill_amount>1500,"High").when(bills.bill_amount>1000,"Medium").otherwise("Low")).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_category|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       Medium|
|      2|             2|        800| Credit Card|          Paid|          Low|
|      3|             3|       1500|        Cash|          Paid|       Medium|
|      4|             4|        900|         UPI|       Pending|          Low|
|      5|             5|       1300|  Debit Card|          Paid|       Medium|
|      6|             6|       2000| Credit Card|          Paid|         High|
|      7|             7|        850|        Cash|     Cancelled|          Low|
|      8|             8|       1400|         UPI|          Paid|       Medium|
|      9|             9|        800|         UPI|          Paid|          Low|
|     10|            10|       1500| Credit Card|   

In [102]:
doctors.withColumn("fee_category",when(doctors.consultation_fee>1000,"Premium").when(doctors.consultation_fee>500,"Standard").otherwise("Basic")).show()

+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_category|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|     Premium|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|    Standard|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|     Premium|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|    Standard|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|     Premium|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|     Premium|
|      107|   Dr Verma|   Dermatology|     Pune|             850|    Standard|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|     Premium|
+---------+-----------+--------------+---------+----------------+------------+



In [103]:
appointments.withColumn("priority",when(appointments.status=="Pending","High").when(appointments.status=="Completed","Normal").otherwise("Low")).show()

+--------------+----------+---------+----------------+---------+--------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|priority|
+--------------+----------+---------+----------------+---------+--------+
|             1|         1|      101|      2024-03-01|Completed|  Normal|
|             2|         2|      102|      2024-03-01|Completed|  Normal|
|             3|         3|      103|      2024-03-02|Completed|  Normal|
|             4|         4|      104|      2024-03-02|  Pending|    High|
|             5|         5|      106|      2024-03-03|Completed|  Normal|
|             6|         6|      105|      2024-03-03|Completed|  Normal|
|             7|         7|      107|      2024-03-04|Cancelled|     Low|
|             8|         8|      108|      2024-03-04|Completed|  Normal|
|             9|         9|      102|      2024-03-05|Completed|  Normal|
|            10|        10|      103|      2024-03-05|Completed|  Normal|
|            11|        11|      101| 

In [104]:
patients.withColumn("city_zone",when(patients.city.isin(["Hyderabad","Bangalore","Chennai"]),"South").otherwise("Metro")).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|city_zone|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    South|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    South|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Metro|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Metro|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    South|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    South|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|    Metro|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Metro|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    South|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|    Metro|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    South|
|        12|Simran|    Delhi| 25|F

---

## Section 9 — Date Functions

76. Convert registration_date to date type.
77. Extract registration year.
78. Extract registration month.
79. Convert appointment_date to date type.
80. Extract appointment month.
81. Count appointments by appointment date.
82. Count appointments by appointment month.
83. Find patients registered after 2023-06-01 .
84. Find days since patient registration.
85. Find days since appointment date.

---

In [106]:
patients.withColumn("registration_date",to_date(patients.registration_date)).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [107]:
patients.select(
    year(patients.registration_date).alias("RegisterationYear"),
    month(patients.registration_date).alias("RegisterationMonth")
).show()

+-----------------+------------------+
|RegisterationYear|RegisterationMonth|
+-----------------+------------------+
|             2023|                 1|
|             2023|                 2|
|             2023|                 3|
|             2023|                 4|
|             2023|                 5|
|             2023|                 6|
|             2023|                 6|
|             2023|                 7|
|             2023|                 7|
|             2023|                 8|
|             2023|                 8|
|             2023|                 8|
+-----------------+------------------+



In [108]:
appointments.withColumn("appointment_date",to_date(appointments.appointment_date)).show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [109]:
appointments.select(
    month(appointments.appointment_date).alias("AppointmentMonth")
).show()

+----------------+
|AppointmentMonth|
+----------------+
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
|               3|
+----------------+



In [113]:
appointments.groupBy("appointment_date").count().show()

+----------------+-----+
|appointment_date|count|
+----------------+-----+
|      2024-03-05|    2|
|      2024-03-06|    2|
|      2024-03-08|    2|
|      2024-03-04|    2|
|      2024-03-02|    2|
|      2024-03-01|    2|
|      2024-03-03|    2|
|      2024-03-07|    2|
+----------------+-----+



In [117]:
appointments.groupBy(month(appointments.appointment_date).alias("AppointmentMonth")).count().show()

+----------------+-----+
|AppointmentMonth|count|
+----------------+-----+
|               3|   16|
+----------------+-----+



In [118]:
patients.filter(patients.registration_date>"2023-06-01").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [119]:
patients.select(
    datediff(current_date(),patients.registration_date).alias("DaysSinceRegistration")
).show()

+---------------------+
|DaysSinceRegistration|
+---------------------+
|                 1203|
|                 1170|
|                 1140|
|                 1108|
|                 1082|
|                 1052|
|                 1040|
|                 1022|
|                 1017|
|                 1000|
|                  991|
|                  980|
+---------------------+



In [120]:
appointments.select(
    datediff(current_date(),appointments.appointment_date).alias("DaysSinceAppointment")
).show()

+--------------------+
|DaysSinceAppointment|
+--------------------+
|                 787|
|                 787|
|                 786|
|                 786|
|                 785|
|                 785|
|                 784|
|                 784|
|                 783|
|                 783|
|                 782|
|                 782|
|                 781|
|                 781|
|                 780|
|                 780|
+--------------------+



---

## Section 10 — Window Functions

86. Rank patients by age within city.
87. Use row_number to find oldest patient per city.
88. Use dense_rank to rank doctors by fee within specialization.
89. Find highest fee doctor per specialization.
90. Find top 2 oldest patients per city.
91. Rank doctors by consultation fee overall.
92. Rank cities by patient count.
93. Rank doctors by appointment count.
94. Create running total bill amount by payment mode.
95. Create running appointment count by doctor.

---

In [121]:
from pyspark.sql.window import Window

In [132]:
w=Window.partitionBy(patients.city).orderBy(patients.age)
patients.withColumn("age_rank",rank().over(w)).show()

+----------+------+---------+---+------+-----------------+--------+
|patient_id|  name|     city|age|gender|registration_date|age_rank|
+----------+------+---------+---+------+-----------------+--------+
|         9| Divya|Bangalore| 33|Female|       2023-07-15|       1|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|       2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|       1|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|       1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|       2|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|       3|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|       1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|       2|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|       3|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|       1|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|       2|
|         7|  Amit|     Pune| 45|  Male|       2

In [129]:
patients.withColumn("row_num",row_number().over(w)).filter("row_num==1").show()

+----------+------+---------+---+------+-----------------+-------+
|patient_id|  name|     city|age|gender|registration_date|row_num|
+----------+------+---------+---+------+-----------------+-------+
|         9| Divya|Bangalore| 33|Female|       2023-07-15|      1|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|      1|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|      1|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|      1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|      1|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|      1|
+----------+------+---------+---+------+-----------------+-------+



In [130]:
w=Window.partitionBy(doctors.specialization).orderBy(doctors.consultation_fee)
doctors.withColumn("fee_rank",dense_rank().over(w)).show()

+---------+-----------+--------------+---------+----------------+--------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_rank|
+---------+-----------+--------------+---------+----------------+--------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|       1|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|       2|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       1|
|      107|   Dr Verma|   Dermatology|     Pune|             850|       2|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|       1|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|       1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|       2|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|       1|
+---------+-----------+--------------+---------+----------------+--------+



In [135]:
w=Window.partitionBy(patients.city).orderBy(patients.age.desc())

In [136]:
patients.withColumn("rank",rank().over(w)).filter("rank<=2").show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+-----------------+----+



In [137]:
w=Window.orderBy(doctors.consultation_fee)
doctors.withColumn("doctor_rank",rank().over(w)).show()

+---------+-----------+--------------+---------+----------------+-----------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|doctor_rank|
+---------+-----------+--------------+---------+----------------+-----------+
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|          1|
|      107|   Dr Verma|   Dermatology|     Pune|             850|          2|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|          3|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|          4|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|          5|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|          6|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|          7|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|          8|
+---------+-----------+--------------+---------+----------------+-----------+



In [140]:
w=Window.orderBy("count")
patients.groupBy("city").count().withColumn("city_rank",rank().over(w)).show()

+---------+-----+---------+
|     city|count|city_rank|
+---------+-----+---------+
|  Chennai|    1|        1|
|     Pune|    1|        1|
|Bangalore|    2|        3|
|   Mumbai|    2|        3|
|    Delhi|    3|        5|
|Hyderabad|    3|        5|
+---------+-----+---------+



In [141]:
appointments.groupBy("doctor_id").count().withColumn("doctor_rank",rank().over(w)).show()

+---------+-----+-----------+
|doctor_id|count|doctor_rank|
+---------+-----+-----------+
|      107|    1|          1|
|      108|    2|          2|
|      103|    2|          2|
|      102|    2|          2|
|      105|    2|          2|
|      106|    2|          2|
|      104|    2|          2|
|      101|    3|          8|
+---------+-----+-----------+



In [145]:
w=Window.partitionBy(bills.payment_mode).orderBy(bills.bill_amount)
bills.withColumn("running_total",sum("bill_amount").over(w)).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|running_total|
+-------+--------------+-----------+------------+--------------+-------------+
|      7|             7|        850|        Cash|     Cancelled|          850|
|     12|            12|        900|        Cash|          Paid|         1750|
|     15|            15|       1200|        Cash|     Cancelled|         2950|
|      3|             3|       1500|        Cash|          Paid|         4450|
|      2|             2|        800| Credit Card|          Paid|          800|
|     10|            10|       1500| Credit Card|          Paid|         2300|
|      6|             6|       2000| Credit Card|          Paid|         6300|
|     13|            13|       2000| Credit Card|          Paid|         6300|
|      5|             5|       1300|  Debit Card|          Paid|         2600|
|     16|            16|       1300|  Debit Card|   

In [151]:
w=Window.partitionBy(appointments.doctor_id).orderBy(appointments.appointment_date)
appointments.withColumn("running_count",count("appointment_id").over(w)).show()

+--------------+----------+---------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|running_count|
+--------------+----------+---------+----------------+---------+-------------+
|             1|         1|      101|      2024-03-01|Completed|            1|
|            11|        11|      101|      2024-03-06|  Pending|            2|
|            15|         6|      101|      2024-03-08|Cancelled|            3|
|             2|         2|      102|      2024-03-01|Completed|            1|
|             9|         9|      102|      2024-03-05|Completed|            2|
|             3|         3|      103|      2024-03-02|Completed|            1|
|            10|        10|      103|      2024-03-05|Completed|            2|
|             4|         4|      104|      2024-03-02|  Pending|            1|
|            12|        12|      104|      2024-03-06|Completed|            2|
|             6|         6|      105|      2024-03-0

---

## Section 11 — RDD Exercises

Create RDD:

```
logs = sc.textFile("hospital_logs.txt")
```

96. Count total log records.
97. Extract user names.
98. Extract actions.
99. Find unique users.
100. Count each action using map and reduceByKey .

---

In [153]:
sc=spark.sparkContext

In [154]:
logs = sc.textFile("hospital_logs.txt")

In [155]:
logs.count()

20

In [159]:
logs.map(lambda x:x.split(" ")[0]).collect()

['Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Aarav',
 'Kiran',
 'Meera',
 'Vikram',
 'Divya',
 'Farhan',
 'Simran',
 'Neha',
 'Amit',
 'Rahul',
 'Meera',
 'Aarav',
 'Priya',
 'Divya',
 'Vikram',
 'Farhan']

In [160]:
logs.map(lambda x:x.split(" ")[1]).collect()

['login',
 'login',
 'appointment',
 'login',
 'payment',
 'appointment',
 'login',
 'appointment',
 'payment',
 'login',
 'appointment',
 'payment',
 'login',
 'payment',
 'appointment',
 'logout',
 'payment',
 'login',
 'payment',
 'appointment']

In [161]:
logs.map(lambda x:x.split(" ")[0]).distinct().collect()

['Kiran',
 'Vikram',
 'Divya',
 'Simran',
 'Amit',
 'Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Meera',
 'Farhan',
 'Neha']

In [163]:
logs.map(lambda x:(x.split(" ")[1],1)).reduceByKey(lambda a,b:a+b).collect()

[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]

---

## Bonus: Complex JSON Exercises

Use patient_profiles.json .

1. Read the JSON file.
2. Print schema.
3. Extract patient_id, name, email, and phone.
4. Explode allergies.
5. Count patients by allergy.
6. Find patients with Dust allergy.
7. Find unique allergies.
8. Join profiles with patients data.
9. Show patient name, city, email, and allergy.
10. Count allergies per city.

---

In [164]:
profiles.show()

+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|       [Pollen]|{priya@mail.com, ...| Priya|         2|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
|      [Seafood]|{meera@mail.com, ...| Meera|         6|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+



In [165]:
profiles.printSchema()

root
 |-- allergies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- patient_id: long (nullable = true)



In [167]:
profiles.select(profiles.patient_id,profiles.name,profiles.contact["email"],profiles.contact["phone"]).show()

+----------+------+---------------+-------------+
|patient_id|  name|  contact.email|contact.phone|
+----------+------+---------------+-------------+
|         1| Aarav| aarav@mail.com|   9000011111|
|         2| Priya| priya@mail.com|   9000022222|
|         3| Rahul| rahul@mail.com|   9000033333|
|         6| Meera| meera@mail.com|   9000066666|
|        10|Vikram|vikram@mail.com|   9000101010|
+----------+------+---------------+-------------+



In [169]:
profiles.select(explode(profiles.allergies).alias("Alergies")).show()

+--------+
|Alergies|
+--------+
|    Dust|
| Peanuts|
|  Pollen|
|    Dust|
|    Milk|
| Seafood|
|  Pollen|
|    Dust|
+--------+



In [170]:
profiles.select(explode(profiles.allergies).alias("Alergies")).groupBy("Alergies").count().show()

+--------+-----+
|Alergies|count|
+--------+-----+
|    Milk|    1|
|    Dust|    3|
| Peanuts|    1|
|  Pollen|    2|
| Seafood|    1|
+--------+-----+



In [175]:
profiles.select(profiles.patient_id,profiles.name,explode(profiles.allergies).alias("Alergies")).filter(col("Alergies")=="Dust").show()

+----------+------+--------+
|patient_id|  name|Alergies|
+----------+------+--------+
|         1| Aarav|    Dust|
|         3| Rahul|    Dust|
|        10|Vikram|    Dust|
+----------+------+--------+



In [176]:
profiles.select(explode(profiles.allergies).alias("Alergies")).distinct().show()

+--------+
|Alergies|
+--------+
|    Milk|
|    Dust|
| Peanuts|
|  Pollen|
| Seafood|
+--------+



In [177]:
profiles.join(patients,profiles.patient_id==patients.patient_id).show()

+---------------+--------------------+------+----------+----------+------+---------+---+------+-----------------+
|      allergies|             contact|  name|patient_id|patient_id|  name|     city|age|gender|registration_date|
+---------------+--------------------+------+----------+----------+------+---------+---+------+-----------------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|       [Pollen]|{priya@mail.com, ...| Priya|         2|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|      [Seafood]|{meera@mail.com, ...| Meera|         6|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+---------------+--------------------+------+----------+----------+------+---------+---+

In [185]:
profiles.join(patients,profiles.patient_id==patients.patient_id).select(patients.name,patients.city,explode(profiles.allergies).alias("Alergies")).groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|   Mumbai|    4|
|Hyderabad|    3|
+---------+-----+



---

## Bonus: Spark SQL Exercises

Create temp views:

```
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")
```

Tasks:

1. Show all patients using SQL.
2. Show patients from Hyderabad.
3. Count patients by city.
4. Count doctors by specialization.
5. Join patients and appointments.
6. Join doctors and appointments.
7. Join appointments and bills.
8. Find total bill amount by payment mode.
9. Rank doctors by consultation fee using SQL.
10. Find top billing patient using SQL.

---


In [186]:
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")

In [187]:
spark.sql("select * from patients").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [188]:
spark.sql("select * from patients where city='Hyderabad'").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [189]:
spark.sql("select city,count(*) from patients group by city").show()

+---------+--------+
|     city|count(1)|
+---------+--------+
|Bangalore|       2|
|  Chennai|       1|
|   Mumbai|       2|
|     Pune|       1|
|    Delhi|       3|
|Hyderabad|       3|
+---------+--------+



In [190]:
spark.sql("select specialization,count(*) from doctors group by specialization").show()

+--------------+--------+
|specialization|count(1)|
+--------------+--------+
|     Neurology|       1|
|   Dermatology|       2|
|    Cardiology|       2|
|    Pediatrics|       1|
|   Orthopedics|       2|
+--------------+--------+



In [191]:
spark.sql("select * from patients join appointments on patients.patient_id=appointments.patient_id").show()

+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|         1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|         2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|         3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|         4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|         5|      106|      2024-03-03|Completed|
|         6| Meera|Hyder

In [192]:
spark.sql("select * from doctors join appointments on doctors.doctor_id=appointments.doctor_id").show()

+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|appointment_id|patient_id|doctor_id|appointment_date|   status|
+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|             1|         1|      101|      2024-03-01|Completed|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|             2|         2|      102|      2024-03-01|Completed|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|             3|         3|      103|      2024-03-02|Completed|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|             4|         4|      104|      2024-03-02|  Pending|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|             5|         5|      

In [193]:
spark.sql("select * from appointments join bills on appointments.appointment_id=bills.appointment_id").show()

+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|             1|         1|      101|      2024-03-01|Completed|      1|             1|       1200|         UPI|          Paid|
|             2|         2|      102|      2024-03-01|Completed|      2|             2|        800| Credit Card|          Paid|
|             3|         3|      103|      2024-03-02|Completed|      3|             3|       1500|        Cash|          Paid|
|             4|         4|      104|      2024-03-02|  Pending|      4|             4|        900|         UPI|       Pending|
|             5|         5|      106|      2024-03-03|Completed|      5|             5|       1300|  Deb

In [194]:
spark.sql("select payment_mode,sum(bill_amount) from bills group by payment_mode").show()

+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+



In [195]:
spark.sql("select rank() over (order by consultation_fee desc) as rank,doctor_name from doctors").show()

+----+-----------+
|rank|doctor_name|
+----+-----------+
|   1|   Dr Mehta|
|   2|    Dr Khan|
|   3|     Dr Rao|
|   4|    Dr Nair|
|   5|  Dr Sharma|
|   6|   Dr Reddy|
|   7|   Dr Verma|
|   8|    Dr Iyer|
+----+-----------+



In [202]:
spark.sql("select * from patients p join appointments a on p.patient_id=a.patient_id join bills b on b.appointment_id=a.appointment_id order by b.bill_amount desc limit 1").show()

+----------+-----+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|patient_id| name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+----------+-----+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|         6|Meera|Hyderabad| 31|Female|       2023-06-10|             6|         6|      105|      2024-03-03|Completed|      6|             6|       2000| Credit Card|          Paid|
+----------+-----+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+

